Version control: Monday, 2026-10-05, 14:40, America/Sao_Paulo (UTC−3)
Luis Anunciacao, PhD

# SOFT follow-up survey: data processing

- Data: private GitHub repository anovabr/school_of_thinkers, `Original data and metadata/`, read with the git credential (gitcreds); SOFT follow-up export (2026-09-27); Escola Modelo (platform demo) excluded.
- Teacher demographics (teacher_id, sex, age, role, training) from the deidentified file `SOFT_teacher_linkage.csv`; no identified data is read.
- Lemmatizes (UDPipe 2.5), scores sentiment (LeIA, Python `leia-br` 0.0.1 via reticulate), attaches the content analysis codes and saves `soft.RData` (`df`, `df_original`) for the DA.
- After a run, `soft.RData` is uploaded to GitHub `Processing and analysis/Importable data/`, where the DA reads it.

### 1. Packages

In [1]:
# load packages; genderBR, reticulate, tm, wordcloud, psych, ordinal, lme4, effectsize, parameters, performance, broom and quanteda.textstats are called with ::
pacman::p_load(tidyverse, stringi, udpipe, igraph, quanteda, rainette)
# data folder of the private GitHub repository and an authenticated handle (token from the git credential store, gitcreds)
gh <- "https://raw.githubusercontent.com/anovabr/school_of_thinkers/main/Original%20data%20and%20metadata/"
gh_auth <- curl::new_handle() |> curl::handle_setheaders(Authorization = paste("token", gitcreds::gitcreds_get()$password))


### 2. Response options of the closed items

In [2]:
# response options of the closed items in instrument order; position in the vector = code 1 to 5
rating_labels <- list(
  ease    = c("Muito difícil", "Difícil", "Razoável", "Fácil", "Muito fácil"),
  freq    = c("Nunca", "Raramente", "Às vezes", "Frequentemente", "Sempre que possível"),
  freq_V  = c("Nunca", "Raramente", "1 ou 2 vezes por semana", "3 ou 4 vezes por semana", "Diariamente"),
  freq_VI = c("Nunca", "Raramente", "Às vezes", "Frequentemente", "Semanalmente"),
  freq_IX = c("Ainda não foi iniciada", "Está sendo organizada", "Está em implementação", "Já funciona parcialmente", "Já funciona plenamente"),
  overall = c("Muito difícil", "Difícil", "Razoável", "Boa", "Muito boa"),
  support = c("Muito insuficiente", "Insuficiente", "Razoável", "Bom", "Muito bom"))

### 3. Raw open answers

In [3]:
# read the raw open answers, one row per answer
df_original <- read_csv(curl::curl(str_c(gh, "SOFT_respostas_abertas_long_2026-09-27.csv"), handle = gh_auth), show_col_types = FALSE)

### 4. Lemmatization (UDPipe)

In [4]:
# lemmatize the open answers (UDPipe 2.5 portuguese-bosque)
df <- df_original |>
  # drop the platform demo school
  filter(Escola != "Escola Modelo") |>
  # section numeral (II to XII) from the section label
  mutate(sec = str_remove(secao, "\\..*"),
         # download the UDPipe model to tempdir() and load it
         lemmas = udpipe_download_model("portuguese-bosque", model_dir = tempdir(),
                                        udpipe_model_repo = "jwijffels/udpipe.models.ud.2.5")$file_model |>
           udpipe_load_model() |>
           # tokenize, tag and lemmatize each answer
           udpipe_annotate(x = resposta, doc_id = seq_along(resposta)) |>
           as_tibble() |>
           # keep nouns, verbs, adjectives and adverbs whose surface form is not a Snowball stopword, >= 2 characters
           filter(upos %in% c("NOUN", "VERB", "ADJ", "ADV"),
                  !stri_trans_general(str_squish(token), "Latin-ASCII; Lower") %in% stri_trans_general(tm::stopwords("portuguese"), "Latin-ASCII; Lower"),
                  str_length(lemma) > 1) |>
           # lemma in lowercase ASCII; doc_id as factor so answers with no lemma are kept
           mutate(lemma = stri_trans_general(str_squish(lemma), "Latin-ASCII; Lower"),
                  doc_id = factor(doc_id, levels = seq_along(resposta))) |>
           # keep alphabetic lemmas only
           filter(str_detect(lemma, "^[a-z]+$")) |>
           # one character vector of lemmas per answer, in answer order
           group_by(doc_id, .drop = FALSE) |>
           summarise(lemmas = list(lemma)) |>
           pull(lemmas))

 - This model has been trained on version 2.5 of data from https://universaldependencies.org



 - The model is distributed under the CC-BY-SA-NC license: https://creativecommons.org/licenses/by-nc-sa/4.0



 - Visit https://github.com/jwijffels/udpipe.models.ud.2.5 for model license details.



 - For a list of all models and their licenses (most models you can download with this package have either a CC-BY-SA or a CC-BY-SA-NC license) read the documentation at ?udpipe_download_model. For building your own models: visit the documentation by typing vignette('udpipe-train', package = 'udpipe')



### 5. English translation of the lemmas

In [5]:
# translate the lemmas to English
df <- df |>
  # lookup lemma -> English from the translation file; Portuguese kept when untranslated
  mutate(lemmas_en = read_csv(curl::curl(str_c(gh, "SOFT_lemma_translation_en.csv"), handle = gh_auth), show_col_types = FALSE) |>
           deframe() |>
           (\(en) map(lemmas, \(l) coalesce(unname(en[l]), l)))())

### 6. Sentiment (LeIA)

In [6]:
# score sentiment per answer (LeIA compound, Python leia-br via reticulate)
# declare the Python dependency; reticulate installs leia-br in its own Python environment (restart the kernel if Python was already started)
reticulate::py_require("leia-br==0.0.1")
df <- df |>
  # one LeIA analyzer, applied to every answer
  mutate(sentiment = reticulate::import("LeIA")$SentimentIntensityAnalyzer() |>
           (\(sia) map_dbl(resposta, \(t) sia$polarity_scores(t)$compound))(),
         # LeIA class: negative <= -.05 < neutral < .05 <= positive
         leia_class = factor(case_when(sentiment <= -.05 ~ "negative", sentiment < .05 ~ "neutral", .default = "positive"),
                             levels = c("negative", "neutral", "positive")))

### 7. Closed ratings of the tools

In [7]:
# add the closed ratings (frequency and ease) of tools II to IX
df <- df |>
  # drop response-level fields; they are joined from the closed-item file later
  select(-Escola, -Turma, -`Data da Resposta`, -`Identificação — Tempo que atua na educação`) |>
  # full join keeps responses without an open answer
  full_join(read_csv(curl::curl(str_c(gh, "EP_professores_acompanhamento_SOFT_2026-09-27.csv"), handle = gh_auth), show_col_types = FALSE) |>
              # drop the platform demo school
              filter(Escola != "Escola Modelo") |>
              # keep the frequency (or stage) and ease items of tools II to IX
              select(id_resposta, matches("^(II|III|IV|V|VI|VII|VIII|IX)\\. .*(frequência|estágio|Como você considera)", ignore.case = FALSE)) |>
              # one row per response, tool and item
              pivot_longer(-id_resposta, names_to = c("sec", "item"), values_to = "label",
                           names_pattern = "^([IVX]+)\\. .*(frequência|estágio|Como você considera).*") |>
              # item name and its response scale; V, VI and IX have their own frequency options
              mutate(item = if_else(item == "Como você considera", "ease", "freq"),
                     scale = case_when(item == "ease" ~ "ease",
                                       sec %in% c("V", "VI", "IX") ~ str_c("freq_", sec),
                                       .default = "freq")) |>
              # response label -> code 1 to 5
              left_join(rating_labels |> enframe("scale", "label") |> unnest_longer(label, indices_to = "code"),
                        by = c("scale", "label")) |>
              # freq and ease as columns
              select(id_resposta, sec, item, code) |>
              pivot_wider(names_from = item, values_from = code) |>
              # tool-major row order
              arrange(fct_inorder(sec)),
            by = c("id_resposta", "sec"))

### 8. Content analysis codes

In [8]:
# add the content analysis codes and the derived rating indicators
df <- df |>
  # mood, implementation and reason of the answers about tools II to IX
  left_join(read_csv(curl::curl(str_c(gh, "SOFT_open_answers_mood_implementation_codes.csv"), handle = gh_auth), show_col_types = FALSE) |>
              transmute(id_resposta, sec = section, mood, implementation, reason),
            by = c("id_resposta", "sec")) |>
  # negative and mixed pooled; top two of ease (easy, very easy) and of frequency
  mutate(mood_group = case_match(mood, c("negative", "mixed") ~ "negative or mixed", .default = mood),
         ease_top2 = as.integer(ease >= 4),
         freq_top2 = as.integer(freq >= 4))

### 9. Teacher and response-level variables

In [9]:
# add the response-level variables: teacher identity and demographics (linkage file), class and program ratings
df <- df |>
  left_join(read_csv(curl::curl(str_c(gh, "EP_professores_acompanhamento_SOFT_2026-09-27.csv"), handle = gh_auth), show_col_types = FALSE) |>
              # drop the platform demo school
              filter(Escola != "Escola Modelo") |>
              # teacher_id, sex, age, role and training from the deidentified linkage file
              left_join(read_csv(curl::curl(str_c(gh, "SOFT_teacher_linkage.csv"), handle = gh_auth), col_types = "iicicc"), by = "id_resposta") |>
              # short names for the overall experience and support items
              rename(overall = matches("^I\\. .*experiência", ignore.case = FALSE),
                     support_school = matches("coordenação pedagógica e pela direção", ignore.case = FALSE),
                     support_coordinator = matches("coordenadora pedagógica", ignore.case = FALSE),
                     support_need = matches("^X\\. .*Existe algum material", ignore.case = FALSE)) |>
              # class evaluated
              mutate(grade = if_else(str_detect(Turma, "^INFANTIL"), "Preschool (final year)", str_c("Grade ", str_sub(Turma, 1, 1))),
                     # time working in education, ordered, in English
                     years_education = factor(`Identificação — Tempo que atua na educação`,
                                              levels = c("até 1 ano", "De 1 a 3 anos", "De 3 a 6 anos", "De 6 a 9 anos", "De 9 a 12 anos", "De 12 a 15 anos", "acima de 15 anos"),
                                              labels = c("Up to 1 year", "1 to 3 years", "3 to 6 years", "6 to 9 years", "9 to 12 years", "12 to 15 years", "More than 15 years")),
                     # overall experience and support labels -> code 1 to 5
                     overall = match(overall, rating_labels$overall),
                     across(c(support_school, support_coordinator), \(x) match(x, rating_labels$support))) |>
              # teachers who answered in both schools
              mutate(school = if (n_distinct(Escola) > 1) "Both schools" else Escola, .by = teacher_id) |>
              # response-level variables kept
              select(id_resposta, teacher_id, Escola, school, sex, age, role, training, grade, years_education,
                     overall, support_school, support_coordinator, support_need),
            by = "id_resposta")

### 10. Section and tool factors

In [10]:
# section and tool as factors in instrument order
df <- df |>
  mutate(sec = factor(sec, levels = c("II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XII")),
         tool = factor(sec, labels = c("Provoking/Questioning Teacher", "Praising Teacher", "Challenging/Thinking Teacher", "Thinkers' Prayer/Phrase",
                                       "Thinkers' Storytellers", "Circle/Double U", "Thinkers' Minute Quiz", "Welcoming Troop/Emotion Firefighters",
                                       "Pedagogical support (X)", "Final testimonials (XII)")))

### 11. Save

In [11]:
# save the DP output for the DA; uploaded to GitHub Processing and analysis/Importable data/soft.RData, which the DA reads
rm(rating_labels, gh, gh_auth)
save(df, df_original, file = "soft.RData")

In [12]:
# size of the analytic data: rows (response x section) and columns
dim(df)

[1] 494  31